<a href="https://colab.research.google.com/github/Chimatanagagopal/RAG/blob/main/HybridRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 11.9 MB/s eta 0:00:00


In [5]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]

reader    = PdfReader(pdf_name)
full_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + "\n"

print(f"Pages      : {len(reader.pages)}")
print(f"Characters : {len(full_text):,}")
print(f"\n--- First 300 chars ---\n")
print(full_text[:300])

Saving PYTHON PROGRAMMING NOTES.pdf to PYTHON PROGRAMMING NOTES.pdf
Pages      : 142
Characters : 139,362

--- First 300 chars ---

PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
 
 
PYTHON PROGRAMMING 
[R17A0554] 
LECTURE NOTES 
B.TECH III YEAR – II SEM (R17) 
(2019-20) 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(


In [6]:
CHUNK_SIZE = 1000
OVERLAP    = 200

chunks = []
start  = 0

while start < len(full_text):
    chunks.append(full_text[start : start + CHUNK_SIZE])
    start += CHUNK_SIZE - OVERLAP

print(f"Total chunks : {len(chunks)}")
print(f"Chunk size   : {CHUNK_SIZE} chars")
print(f"Overlap      : {OVERLAP} chars")

Total chunks : 175
Chunk size   : 1000 chars
Overlap      : 200 chars


In [7]:
i = 0
print(f"=== Chunk {i} ===")
print(f"Length: {len(chunks[i])} chars\n")
print(chunks[i])

=== Chunk 0 ===
Length: 1000 chars

PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
 
 
PYTHON PROGRAMMING 
[R17A0554] 
LECTURE NOTES 
B.TECH III YEAR – II SEM (R17) 
(2019-20) 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 
Certified) 
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India 
 
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
 
 
SYLLABUS 
MALLA REDDY COLLEGE OF ENGINEERING AND TECHNOLOGY 
 
III Year B. Tech CSE -II SEM       L T/P/D C 
             3  - / - / -  3 
OPEN ELECTIVE III 
(R17A0554) PYTHON PROGRAMMING 
OBJECTIVES:  
 To read and write simple Python programs. 
 To

In [8]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.7 MB/s eta 0:00:00


Load Embedding Model

In [9]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")
print("Embedding dimension:", embedder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!
Embedding dimension: 384


/tmp/ipykernel_2036/3420059995.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedder.get_sentence_embedding_dimension())


Embed All Chunks

In [10]:
import numpy as np

embeddings = embedder.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

print("Shape:", embeddings.shape)
print("Meaning: ", embeddings.shape[0], "chunks →", embeddings.shape[1], "dimensions each")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Shape: (175, 384)
Meaning:  175 chunks → 384 dimensions each


In [11]:
import faiss

dim         = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(embeddings)

print("Vectors stored:", faiss_index.ntotal)

Vectors stored: 175


In [12]:
query       = "What is this document about?"
query_embed = embedder.encode([query]).astype("float32")

D, I = faiss_index.search(query_embed, 5)

print("Top 5 matching chunks:\n")
for rank, (dist, idx) in enumerate(zip(D[0], I[0]), start=1):
    print(f"--- Rank {rank} | Distance: {dist:.4f} ---")
    print(chunks[idx][:300])
    print()

Top 5 matching chunks:

--- Rank 1 | Distance: 1.4726 ---
 1, 2: 4, 3: 9, 4: 16, 5: 25} 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
103 
 
UNIT – V 
FILES, EXCEPTIONS, MODULES, PACKAGES 
Files and exception:  text files, reading and writing files, command line arguments, errors 
and exceptions, handli

--- Rank 2 | Distance: 1.5733 ---
plate 
This class is used to create a string template for simpler string substitutions 
Syntax: from string import Template 
Python arrays: 
Array is a container which can hold a fix number of items and these items should be of the 
same type. Most of the data structures make use of arrays to implem

--- Rank 3 | Distance: 1.5762 ---
n: 
Syntax: true_value if Condition else false_value 
>>> x = "1" if True else "2" 
>>> x 
'1' 
Statements: 
A statement is an instruction that the Python interpreter can execute. We have normally two 
basic statements , the assignment statement and the print statement.

In [13]:
!pip install -q rank_bm25

In [14]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25             = BM25Okapi(tokenized_chunks)

print("BM25 index ready!")
print("Documents indexed:", len(tokenized_chunks))

BM25 index ready!
Documents indexed: 175


In [15]:
query       = "What is this document about?"
query_tokens = query.lower().split()

scores      = bm25.get_scores(query_tokens)
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("Top 5 BM25 results:\n")
for rank, idx in enumerate(top_indices, start=1):
    print(f"--- Rank {rank} | BM25 Score: {scores[idx]:.4f} ---")
    print(chunks[idx][:300])
    print()

Top 5 BM25 results:

--- Rank 1 | BM25 Score: 6.0280 ---
 r * r 
    return (c, a) 
print(circleInfo(10)) 
Output: 
C:/Users/MRCET/AppData/Local/Programs/Python/Python38-32/functupretval.py 
(62.8318, 314.159) 
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
96 
 
------------------------------ 
def f(x): 
 

--- Rank 2 | BM25 Score: 5.2119 ---
ython/Python38-32/pyyy/ex3.py 
number can not be divided by zero 
bye 
 The except block executes only when try block has an error, check it below  
a=5 
b=2 
try: 
    print(a/b) 
except Exception: 
    print("number can not be divided by zero") 
    print("bye") 
Output: 
C:/Users/MRCET/AppData/L

--- Rank 3 | BM25 Score: 5.1709 ---
 You are free to choose the 
number of spaces of indentation to use, but you then need to stick with it. 
Syntax errors: 
These are the most basic type of  error. They arise when the  Python parser   is unable to 
understand a line of code.  Syntax errors are almost 

In [16]:
query = "What is this document about?"

query_embed  = embedder.encode([query]).astype("float32")
_, dense_I   = faiss_index.search(query_embed, 5)
dense_idx    = dense_I[0].tolist()

query_tokens = query.lower().split()
scores       = bm25.get_scores(query_tokens)
sparse_idx   = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("Dense  top-5 indices:", dense_idx)
print("Sparse top-5 indices:", sparse_idx)
print()
print("Overlap (found by both):", set(dense_idx) & set(sparse_idx))

Dense  top-5 indices: [129, 95, 26, 2, 37]
Sparse top-5 indices: [120, 148, 143, 20, 35]

Overlap (found by both): set()


In [17]:
def dense_search(query, top_k=10):
    q_emb = embedder.encode([query]).astype("float32")
    _, I  = faiss_index.search(q_emb, top_k)
    return I[0].tolist()


def sparse_search(query, top_k=10):
    tokens  = query.lower().split()
    scores  = bm25.get_scores(tokens)
    indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return indices[:top_k]


def reciprocal_rank_fusion(dense_indices, sparse_indices, k=60):
    scores = {}

    for rank, idx in enumerate(dense_indices, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    for rank, idx in enumerate(sparse_indices, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    return sorted(scores, key=scores.get, reverse=True)

In [18]:
query = "What is this document about?"

dense_idx  = dense_search(query,  top_k=10)
sparse_idx = sparse_search(query, top_k=10)
fused_idx  = reciprocal_rank_fusion(dense_idx, sparse_idx)

print("Dense  top-10:", dense_idx)
print("Sparse top-10:", sparse_idx)
print("Fused  top-10:", fused_idx[:10])
print()
print("Chunks that appeared in BOTH:", set(dense_idx) & set(sparse_idx))

Dense  top-10: [129, 95, 26, 2, 37, 1, 6, 44, 3, 133]
Sparse top-10: [120, 148, 143, 20, 35, 146, 139, 141, 37, 129]
Fused  top-10: [129, 37, 120, 95, 148, 26, 143, 2, 20, 35]

Chunks that appeared in BOTH: {129, 37}


In [19]:
print("Top 5 chunks after RRF:\n")
for rank, idx in enumerate(fused_idx[:5], start=1):
    in_dense  = "✅" if idx in dense_idx  else "❌"
    in_sparse = "✅" if idx in sparse_idx else "❌"
    print(f"--- Rank {rank} | Dense:{in_dense} Sparse:{in_sparse} ---")
    print(chunks[idx][:300])
    print()

Top 5 chunks after RRF:

--- Rank 1 | Dense:✅ Sparse:✅ ---
 1, 2: 4, 3: 9, 4: 16, 5: 25} 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
103 
 
UNIT – V 
FILES, EXCEPTIONS, MODULES, PACKAGES 
Files and exception:  text files, reading and writing files, command line arguments, errors 
and exceptions, handli

--- Rank 2 | Dense:✅ Sparse:✅ ---
nt is used to exit a function and go back to the place from where it was 
called. This statement can contain expression which gets evaluated and the value is returned. 
If there is no expression in the statement or the  return statement itself is not present inside a 
function, then the function wil

--- Rank 3 | Dense:❌ Sparse:✅ ---
 r * r 
    return (c, a) 
print(circleInfo(10)) 
Output: 
C:/Users/MRCET/AppData/Local/Programs/Python/Python38-32/functupretval.py 
(62.8318, 314.159) 
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
96 
 
-------------

In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Reranker loaded!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker loaded!


In [21]:
def rerank(query, candidate_indices, top_n=5):
    pairs  = [(query, chunks[i]) for i in candidate_indices]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidate_indices, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return [(idx, score) for idx, score in ranked[:top_n]]


query      = "What is this document about?"
candidates = fused_idx[:10]
reranked   = rerank(query, candidates)

print("After Reranking:\n")
for rank, (idx, score) in enumerate(reranked, start=1):
    print(f"--- Rank {rank} | Reranker Score: {score:.4f} ---")
    print(chunks[idx][:300])
    print()

After Reranking:

--- Rank 1 | Reranker Score: -4.1281 ---
 1, 2: 4, 3: 9, 4: 16, 5: 25} 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
103 
 
UNIT – V 
FILES, EXCEPTIONS, MODULES, PACKAGES 
Files and exception:  text files, reading and writing files, command line arguments, errors 
and exceptions, handli

--- Rank 2 | Reranker Score: -7.3407 ---
quely identify it. Functio n naming follows the same  rules of writing 
identifiers in Python. 
3. Parameters (arguments) through which we pass values to a function. They are optional. 
4. A colon (:) to mark the end of function header. 
5. Optional documentation string (docstring) to describe what 

--- Rank 3 | Reranker Score: -7.7258 ---
n: 
Syntax: true_value if Condition else false_value 
>>> x = "1" if True else "2" 
>>> x 
'1' 
Statements: 
A statement is an instruction that the Python interpreter can execute. We have normally two 
basic statements , the assignment statement and the p